# 4. Choosing the right note

We now have the sung notes (pitch with cents). For each one, the app must decide its **target**: the note it
should be. This is the heart of the app, and it is a decision problem, not signal processing.

There are two very different kinds of mistakes:

* **Out of tune**: the singer aimed at the right note but missed by some cents (a C sung 30 cents sharp).
  Fix: move it back to the nearest good note.
* **Wrong note**: the singer sang a different note than the melody, sometimes *accurately*. No amount of
  "nearest note" correction can fix this; the app has to infer what the melody *should* be.

This notebook goes from the simplest rule to the most context-aware one:

1. nearest semitone,
2. chord- and key-aware choice with a linear cost,
3. strict key mode,
4. PHRASE: decode a whole phrase with Viterbi (dynamic programming),
5. REPETITION: compare with the other verses,
6. repeats in a new key (after a modulation),
7. how strongly to move.

All of this is in `melody.py`; `pitch.plan_corrections` turns the targets into shifts.

In [1]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")  # hide library chatter (progress bars etc.)
import numpy as np
import matplotlib.pyplot as plt
import librosa
from nbtools import *   # synthetic sounds, play(), plots (see nbtools.py)
plt.rcParams["figure.dpi"] = 90

from songpolish.analysis import HARMONY_HOP, HarmonyMap, Key, KeySpan, harmony_map
from songpolish.melody import SungNote, TargetStrategy, choose_targets, note_cost
A_MAJOR = Key(tonic=9, minor=False)

def notes_at(pitches, start_s=0.0, note_s=0.4, gap_s=0.1):
    # hand-made SungNotes: we skip pitch detection to focus on the decision
    out, t = [], start_s
    for p in pitches:
        out.append(SungNote(int(t * SR), int((t + note_s) * SR), p, np.full(30, p)))
        t += note_s + gap_s
    return out

def a_chord_harmony(seconds, rising_line=False):
    # a 'harmony map' where the guitar plays an A major chord all the time
    frames = int(seconds * SR / HARMONY_HOP) + 1
    chord = np.full(12, 0.1); chord[[9, 1, 4]] = [1.0, 0.8, 0.8]          # A, C#, E
    line = np.linspace(45, 45 + 4 * seconds, frames) if rising_line else np.full(frames, 50.0)
    return HarmonyMap(chroma=np.repeat(chord[:, None], frames, axis=1), profile=chord, sr=SR,
                      key=A_MAJOR, pitch_line=line)

names = lambda ts: [note_name(t) for t in ts]

## 4.1 The simplest rule: nearest semitone

Round each pitch to the nearest integer MIDI number (relative to the guitar's tuning). The error is
`target − pitch`, in semitones (× 100 = cents).

This is what classic "auto-tune" does. It fails exactly when the singer is far off: a note sung 55 cents
sharp rounds to the *next* semitone, which is usually a wrong note.

In [2]:
sung = [57.1, 60.55, 64.2]
print("sung:   ", [f"{p:.2f}" for p in sung])
print("nearest:", names(np.round(sung)), " <- 60.55 becomes C#, but over an A chord C# is fine; over a C chord it would clash")

sung:    ['57.10', '60.55', '64.20']
nearest: ['A3', 'C#4', 'E4']  <- 60.55 becomes C#, but over an A chord C# is fine; over a C chord it would clash


## 4.2 Listening to the guitar: a cost per candidate

The real singer this was tuned on sat 20–30 cents sharp of the guitar with wide scatter, so "nearest
semitone" put about 35% of notes on out-of-key neighbours (an A# over an A chord). The app instead scores
each candidate note $k$ within 1 semitone of the sung pitch $p$:

$$\text{cost}(k) = \frac{|k - p|}{0.35} \;-\; 1.5 \cdot \text{chord weight}(k) \;-\; 1.5 \cdot [k \text{ in key}]$$

and picks the cheapest. The terms:

* **distance**: moving far costs (one unit per 35 cents),
* **chord weight**: from the chroma during the note (notebook 2), 0..1. Chord tones are rewarded,
* **in key**: notes of the current key get a bonus.

Consequences: an off-chord note goes to the chord tone if the singer is more than ~35 cents away from it,
and an out-of-key note is kept only if it was sung within roughly 20–25 cents (the exact break-even depends
on the chord weights). Then it is probably intentional, e.g. a blue note.

Why **linear** and not squared distance (as in least squares)? With a squared cost, a note 0.4 st from an
off-chord neighbour and 0.6 st from the chord tone costs $0.16$ vs $0.36$, and no realistic chord bonus
can overcome that. The chord never wins. Linear costs make the trade-off proportionate.

The case that motivated the key bonus: **notes around 236–238 Hz sung over an A chord**. 236 Hz is A# + 22
cents, and 238 Hz is A# + 36 cents. Nearest semitone says A# for both, but A# clashes with A major; a flat B
is far more likely.

In [3]:
h = a_chord_harmony(3)
w = h.weights(0, SR)
for hz in (236, 238):
    p = 69 + 12 * np.log2(hz / 440)                       # Hz -> MIDI number
    print(f"{hz} Hz = {note_name(p)} {100 * (p - round(p)):+.0f} cents. Candidates within 1 semitone:")
    for k in [58, 59]:
        print(f"   {note_name(k):3s}: distance {abs(k - p):.2f} st, chord weight {w[k % 12]:.2f}, "
              f"in key {k % 12 in A_MAJOR.pitch_classes!s:5s} -> cost {note_cost(p, k, w, A_MAJOR.pitch_classes):+.2f}")
    print("   chosen:", names(choose_targets(notes_at([p]), TargetStrategy.NEAREST, h)),
          "| strict key mode:", names(choose_targets(notes_at([p]), TargetStrategy.NEAREST, h, [A_MAJOR.pitch_classes])))
# break-even: d/0.35 = (1-d)/0.35 - 1.5 (both chord weights 0.1)  ->  d = (1 - 0.525) / 2
print(f"break-even distance from A#: {(1 - 0.35 * 1.5) / 2 * 100:.0f} cents")

236 Hz = A#3 +22 cents. Candidates within 1 semitone:
   A#3: distance 0.22 st, chord weight 0.10, in key False -> cost +0.47
   B3 : distance 0.78 st, chord weight 0.10, in key True  -> cost +0.59
   chosen: ['A#3'] | strict key mode: ['B3']
238 Hz = A#3 +36 cents. Candidates within 1 semitone:
   A#3: distance 0.36 st, chord weight 0.10, in key False -> cost +0.88
   B3 : distance 0.64 st, chord weight 0.10, in key True  -> cost +0.17
   chosen: ['B3'] | strict key mode: ['B3']
break-even distance from A#: 24 cents


So the rule is a threshold, as designed: 236 Hz is 22 cents from A#, just inside the 24-cent break-even, and is
kept as a (possibly intentional) A#; 238 Hz is 36 cents away and becomes B. Strict key mode never keeps A#.

## 4.3 Strict key mode

Lesson from the first real user: wrong notes are often sung **accurately**. That singer's scale steps were
a semitone too narrow: E–F–F#–G–A over A major instead of E–F#–G#–A. The chord-aware rule keeps an accurate
out-of-key note as "intentional", so it kept exactly those errors.

**Strict key mode** (`--pitch-mode key`) removes the choice: only notes of the current key are allowed
candidates. The musician preferred this on that song. It is not the default, because songs that use
notes outside the key on purpose (blues, chromatic passing notes) would be "corrected" into blandness.

In [4]:
run = [64.0, 65.05, 66.0, 67.0, 69.0]                 # E F F# G A, each sung accurately
notes = notes_at(run)
allowed = [A_MAJOR.pitch_classes] * len(notes)
print("sung:             ", names(run))
print("chord-aware:      ", names(choose_targets(notes, TargetStrategy.NEAREST, h)))
print("strict key mode:  ", names(choose_targets(notes, TargetStrategy.NEAREST, h, allowed)))

sung:              ['E4', 'F4', 'F#4', 'G4', 'A4']
chord-aware:       ['E4', 'F4', 'F#4', 'G4', 'A4']
strict key mode:   ['E4', 'F#4', 'F#4', 'F#4', 'A4']


With strict mode the F and the G, accurately sung but not in A major, both move to F#. Each had two in-key
neighbours one semitone away (E/F# and F#/G#), and over this chord the tie goes to F#. In PHRASE mode the
melody decides such ties: a rising line would choose E–F#–G#–A.

## 4.4 PHRASE: decide a whole phrase at once (Viterbi)

Deciding notes one by one ignores the **melody**. A musician hears that a line should go up, and the real
complaint was exactly that: "the melody goes high-low-high where the guitar goes low-mid-high".

PHRASE treats each phrase (notes separated by less than 0.6 s) as a **sequence labelling** problem:

* **states**: for each note, its candidate targets. Notes that **fit** (chord tone and in key) are anchors
  with only one candidate; notes that **clash** may move up to 3 semitones, to in-key notes only,
* **emission cost**: the per-note cost above, plus a penalty for moving more than 1 semitone,
* **transition cost** between consecutive targets:
  * leaps larger than a whole tone cost extra (melodies mostly move by steps),
  * moving **against the guitar's direction** costs 3.0. The "guitar line" is the energy-weighted average
    pitch of the guitar over time (`HarmonyMap.line`), which rises when the guitar goes low-mid-high,
  * moving against the direction the singer actually sang costs 1.0.
* **Viterbi** (dynamic programming) finds the cheapest path through the whole phrase in
  O(notes × candidates²).

Example: the guitar climbs; the singer sings A, then **drops** to 54.6 (between F# and G, neither in the A chord), then C#.

In [5]:
notes = notes_at([57.0, 54.6, 61.0])
h_rising = a_chord_harmony(3, rising_line=True)
print("sung:    ", names([n.pitch for n in notes]))
print("NEAREST: ", names(choose_targets(notes, TargetStrategy.NEAREST, h_rising)), " (keeps the drop)")
print("PHRASE:  ", names(choose_targets(notes, TargetStrategy.PHRASE, h_rising)), " (moves with the guitar, onto the chord tone)")

sung:     ['A3', 'G3', 'C#4']
NEAREST:  ['A3', 'F#3', 'C#4']  (keeps the drop)
PHRASE:   ['A3', 'A3', 'C#4']  (moves with the guitar, onto the chord tone)


## 4.5 REPETITION: songs repeat themselves

Choruses and verses come back. If the singer sings one note differently in one verse while the other
verses agree, the outlier is probably the mistake. REPETITION:

1. describes each note by its **context**: the chord weights and the sung pitch classes in ±2 s, as one unit
   vector (a hand-made embedding),
2. finds notes at least 6 s away with **cosine similarity** ≥ 0.9 (the same place in another verse),
3. lines the phrases up using the neighbouring notes, one match per repetition,
4. if at least 2 repetitions (and ≥ 60%) agree on a different note within 3.5 semitones, moves the note there.

This is a k-nearest-neighbours vote. Example: the same phrase sung four times; in verse 3 the top note is
C# instead of E.

In [6]:
PHRASE = [57.0, 59.0, 61.0, 64.0, 61.0, 59.0, 57.0]     # A B C# E C# B A
notes = []
for verse, t0 in enumerate([0, 10, 20, 30]):
    pitches = list(PHRASE)
    if verse == 2: pitches[3] = 61.1                    # the mistake
    notes += notes_at(pitches, start_s=t0)
targets = choose_targets(notes, TargetStrategy.REPETITION, a_chord_harmony(34))
wrong = 2 * len(PHRASE) + 3
print("verse 3 sung:   ", names([n.pitch for n in notes[14:21]]))
print("verse 3 target: ", names(targets[14:21]), f"  <- note {wrong} follows the other verses")

verse 3 sung:    ['A3', 'B3', 'C#4', 'C#4', 'C#4', 'B3', 'A3']
verse 3 target:  ['A3', 'B3', 'C#4', 'E4', 'C#4', 'B3', 'A3']   <- note 17 follows the other verses


## 4.6 Repeats in a new key

The real song changes key at the end: the last chorus is the earlier chorus **3 semitones higher** (A major
→ C major). But the singer only went about 1 semitone up. Snapping every note to the nearest C-major note
put them in key, yet several were a scale step too low: the musician heard a drop where the melody rises.

The fix combines PHRASE/REPETITION with the key changes of notebook 2. For each phrase after a key change:

1. **find its earlier repetition** in another key: slide the phrase's sung pitch contour (sampled every 50 ms)
   over the earlier part of the song, allowing any transposition (compare after removing the median
   difference), and keep the best match if the mean difference is below 0.8 semitones,
2. compute the **interval** between the two key signatures (A major → C major: +3),
3. **ask the guitar to confirm** it: correlate the earlier accompaniment's chroma, shifted by each of the 12
   possible intervals, with the new passage. The key-change interval must fit best. Each note's mean level
   is removed first, so a ringing drone (equal in both passages) can't vote for "no transposition".
   A change to the parallel minor (A major → A minor) has the same signature interval but fails this check,
   and correctly keeps the melody where it is,
4. each note takes **its earlier counterpart's target + the interval**, in the octave that was sung.

The synthetic test: a phrase sung in A major, then in a section 3 semitones up, sung only 1 semitone up.

In [7]:
def modulating_harmony(transposed):
    per_s = SR / HARMONY_HOP
    chords = [[9, 1, 4], [2, 6, 9], [4, 8, 11]]                  # A, D, E chords
    frames = []
    for second in range(40):
        c = np.full(12, 0.1); root = chords[second % 3]
        c[[(q + 3) % 12 for q in root] if transposed and second >= 20 else root] = 1.0
        frames += [c] * int(per_s)
    chroma = np.array(frames).T; half = chroma.shape[1] // 2
    return HarmonyMap(chroma=chroma, profile=chroma.mean(1), sr=SR, key=A_MAJOR,
                      pitch_line=np.full(chroma.shape[1], 50.0),
                      keys=(KeySpan(0, 20 * SR, A_MAJOR, chroma[:, :half].mean(1)),
                            KeySpan(20 * SR, 40 * SR, Key(0, False), chroma[:, half:].mean(1))))

MELODY = [57.0, 59.0, 61.0, 62.0, 64.0]
notes = notes_at(MELODY, start_s=2.0) + notes_at([p + 1 for p in MELODY], start_s=22.0)
for transposed in (True, False):
    t = choose_targets(notes, TargetStrategy.PHRASE, modulating_harmony(transposed))
    print(f"guitar transposed {transposed!s:5s}: first chorus {names(t[:5])}, last chorus sung {names([n.pitch for n in notes[5:]])} -> {names(t[5:])}")

guitar transposed True : first chorus ['A3', 'B3', 'C#4', 'D4', 'E4'], last chorus sung ['A#3', 'C4', 'D4', 'D#4', 'F4'] -> ['C4', 'D4', 'E4', 'F4', 'G4']
guitar transposed False: first chorus ['A3', 'B3', 'C#4', 'D4', 'E4'], last chorus sung ['A#3', 'C4', 'D4', 'D#4', 'F4'] -> ['A#3', 'C4', 'D4', 'D#4', 'F4']


With a transposed accompaniment, the last chorus becomes the first one + 3 semitones (C D E F G). Without
it, the app does not trust the key-change interval and falls back to the normal per-note choice. Here
that keeps the sung notes: they were sung accurately, and the default mode keeps accurate notes even
outside the key (A#, D#). In strict key mode they would be moved into C major.

## 4.7 How strongly to move

`plan_corrections` turns each target into one constant shift for the note:

* **fine-tuning** (target = the nearest semitone actually sung): move `strength × error`, with strength
  0.8 by default, so a little of the human imprecision remains. Errors under 8 cents are left alone;
* **wrong-note fixes** (target ≠ nearest sung semitone): move the **full** distance. 80% of a two-semitone
  fix would still be wrong, just differently.

In [8]:
from songpolish.pitch import plan_corrections
melody = voice_like([60.3, 62.0, 64.4], note_s=0.6)
shifts, report = plan_corrections(melody, strength=0.8)
for s in shifts:
    print(f"{s.start / SR:.2f}-{s.end / SR:.2f} s: shift {100 * s.semitones:+.0f} cents")
print(report.summary())

0.00-0.62 s: shift -27 cents
1.34-1.98 s: shift -35 cents
2/3 notes nudged; average error 27 cents, average shift 31 cents.


The in-tune D is left alone; the C (30 cents sharp) and the E (40 cents sharp) move 80% of the way.

## Summary

| strategy | decides with | can fix |
|---|---|---|
| NEAREST (default) | distance + chord + key, per note, ≤ 1 st | out-of-tune notes, off-chord neighbours |
| + strict key | only in-key candidates | accurately sung out-of-key notes |
| PHRASE | Viterbi over the phrase: leaps, guitar direction, sung direction | wrong notes up to 3 st that clash |
| REPETITION | vote of the same spot in other verses | outliers against the rest of the song |
| repeats in a new key (PHRASE/REPETITION) | earlier chorus + key-change interval, confirmed by the guitar | a singer who didn't follow a modulation |

Next: **notebook 5**, how a note is actually moved without making it sound artificial.